
# D2D Hybrid Bonding Warpage Yield Demo

This notebook demonstrates a D2D hybrid-bonding stack warpage calculation using a multilayer stress/force-balance model with thermal strain, Cu residual eigenstrain, and initial die warpage.

The flow is:

$$
\{h_i,E_i,\nu_i,\alpha_i,\epsilon_i^{\mathrm{res}},\Delta T,W_i^0\}_{i=1}^{N}
\longrightarrow
\{\widetilde W_k\}_{k=1}^{N-1}
\longrightarrow
Y_{\mathrm{stack}}^{\mathrm{warp}}
=
\Pr\!\left(\left|\widetilde W_k\right|<W_{\mathrm{th}},\ \forall k\right)
$$

The notebook includes:

1. deterministic total warpage calculation,
2. sensitivity matrix calculation,
3. analytical multivariate-Gaussian yield calculation,
4. Monte Carlo validation,
5. timing records.

Default example:
- 16 dies
- 10 mm × 10 mm die size
- each die unit is 50 µm total thickness
- each die unit repeats the same Si + SiO2 + Cu physical sublayers
- threshold $W_{\mathrm{th}}=20\,\mu\mathrm{m}$


In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import time

np.set_printoptions(precision=4, suppress=True)



## 1. Model implementation

For each physical sublayer, the free strain is modeled as thermal strain plus residual eigenstrain:

$$
\epsilon_i^{\mathrm{free}}
=
\alpha_i\Delta T
+
\epsilon_i^{\mathrm{res}}.
$$

The common in-plane strain component is removed through the stiffness-weighted average. With

$$
\Delta\epsilon_i
=
\epsilon_i^{\mathrm{free}}
-
\epsilon_0^{\mathrm{free}},
$$

the axial force per unit width is

$$
F_i =
\frac{1}{\lambda_i}
\left[
\left(\Delta\epsilon_i-s_{\epsilon}\right)
-
(a_i-s_a)\kappa
+
(b_i-s_b)
\right].
$$

The final curvature is obtained from moment equilibrium:

$$
\sum_{i=0}^{m}F_i a_i
+
\sum_{i=0}^{m}D_i\left(\kappa-\kappa_i^0\right)=0.
$$

Here $W_i^0$ is the signed initial die warpage and is converted to initial curvature by

$$
\kappa_i^0 = \frac{2W_i^0}{L^2}.
$$


In [2]:

def multilayer_warpage(layers, delta_T, L, initial_W=None):
    """
    Compute released signed warpage of a physical multilayer stack.

    Parameters
    ----------
    layers : dict
        Dictionary with arrays:
        h                : layer thickness [m]
        E                : Young's modulus [Pa]
        nu               : Poisson's ratio
        alpha            : CTE [1/K]
        epsilon_residual : residual eigenstrain [-], optional
    delta_T : float
        Temperature change [K]. Use a consistent sign convention.
        Example: final - anneal = 25 - 300 = -275 K.
    L : float
        Half-length of square die [m].
    initial_W : array-like or None
        Signed initial bow/warpage of each physical sublayer before release [m].
        If None, all initial warpages are set to zero.

    Returns
    -------
    result : dict
        W      : signed stack warpage [m]
        kappa  : stack curvature [1/m]
        F      : axial force per unit width in each layer [N/m]
        sigma  : average in-plane stress in each layer [Pa]
        aux    : intermediate terms useful for debugging
    """
    h = np.asarray(layers["h"], dtype=float)
    E = np.asarray(layers["E"], dtype=float)
    nu = np.asarray(layers["nu"], dtype=float)
    alpha = np.asarray(layers["alpha"], dtype=float)
    epsilon_residual = np.asarray(
        layers.get("epsilon_residual", np.zeros_like(h)), dtype=float
    )

    n_layers = len(h)
    if not (len(E) == len(nu) == len(alpha) == len(epsilon_residual) == n_layers):
        raise ValueError("All layer arrays must have the same length.")

    if initial_W is None:
        initial_W = np.zeros(n_layers)
    initial_W = np.asarray(initial_W, dtype=float)
    if len(initial_W) != n_layers:
        raise ValueError("initial_W must have one entry per physical layer.")

    # Initial curvature of each physical sublayer.
    kappa0 = 2.0 * initial_W / (L**2)

    # Biaxial modulus and axial compliance.
    E0 = E / (1.0 - nu)
    lam = 1.0 / (E0 * h)

    # Free strain combines thermal strain and residual eigenstrain.
    free_strain = alpha * delta_T + epsilon_residual
    delta_free_strain = free_strain - free_strain[0]

    # Geometry term a_i.
    cum_h = np.cumsum(h)
    a = cum_h - 0.5 * (h[0] + h)

    # Initial-curvature geometry term b_i.
    b = np.zeros(n_layers)
    for i in range(n_layers):
        b[i] = np.sum(h[: i + 1] * kappa0[: i + 1]) - 0.5 * (
            h[0] * kappa0[0] + h[i] * kappa0[i]
        )

    # Stiffness-weighted averages.
    weights = 1.0 / lam
    denom = np.sum(weights)
    s_free = np.sum(delta_free_strain * weights) / denom
    s_a = np.sum(a * weights) / denom
    s_b = np.sum(b * weights) / denom

    # Layer bending stiffness per unit width.
    D = E0 * h**3 / 12.0

    # Compact terms.
    M_free = np.sum((a / lam) * (delta_free_strain - s_free))
    M_b = np.sum((a / lam) * (b - s_b))
    K_a = np.sum((a / lam) * (a - s_a))
    K_D = np.sum(D)
    K_eff = K_D - K_a
    M_D0 = np.sum(D * kappa0)

    # Final curvature.
    kappa = (M_D0 - M_free - M_b) / K_eff

    # Warpage at half-length L.
    W = kappa * L**2 / 2.0

    # Axial force and stress.
    F = (1.0 / lam) * (
        (delta_free_strain - s_free)
        - (a - s_a) * kappa
        + (b - s_b)
    )
    sigma = F / h

    return {
        "W": W,
        "kappa": kappa,
        "F": F,
        "sigma": sigma,
        "aux": {
            "E0": E0,
            "lambda": lam,
            "free_strain": free_strain,
            "epsilon_residual": epsilon_residual,
            "delta_free_strain": delta_free_strain,
            "a": a,
            "b": b,
            "s_free": s_free,
            "s_a": s_a,
            "s_b": s_b,
            "D": D,
            "M_free": M_free,
            "M_b": M_b,
            "K_a": K_a,
            "K_D": K_D,
            "K_eff": K_eff,
            "M_D0": M_D0,
        },
    }



## 2. Demo parameters

The deterministic example now uses a repeated physical die unit instead of a fitted CTE profile.
Each added die has the same 50 µm total thickness and is represented by the same Si + SiO2 + Cu sublayers.
The Cu layer carries a fixed residual biaxial stress, converted to residual eigenstrain by

$$
\epsilon_{\mathrm{Cu}}^{\mathrm{res}}
=
\frac{\sigma_{\mathrm{Cu}}^{\mathrm{res}}}{M_{\mathrm{Cu}}},
\qquad
M_{\mathrm{Cu}}=\frac{E_{\mathrm{Cu}}}{1-\nu_{\mathrm{Cu}}}.
$$

The stack is simulated sequentially. After each release, the deterministic bow from the previous step is carried into the next bonding step, then the next identical stressed die unit is appended and the full multilayer curvature is recomputed.
This creates the intended trend: bow magnitude accumulates, while the added increment decreases as the stack stiffness increases.

The initial signed die warpage is modeled as independent Gaussian:

$$
W_i^0\sim \mathcal{N}\!\left(\mu_i,\sigma_i^2\right).
$$

All values are example parameters and can be changed.


In [3]:

rng = np.random.default_rng(7)

# Geometry
N_die = 16
die_size = 10e-3      # 10 mm
L = die_size / 2      # half-length = 5 mm
W_th = 20e-6          # 20 um threshold

# Thermal process
T_anneal = 300.0      # degC
T_final = 25.0        # degC
delta_T = T_final - T_anneal

# Repeated die unit: total thickness remains 50 um.
die_total_thickness = 50e-6
sio2_thickness = 1.5e-6
cu_thickness = 0.3e-6
si_thickness = die_total_thickness - sio2_thickness - cu_thickness
if si_thickness <= 0.0:
    raise ValueError("Si thickness must remain positive.")

# Material parameters.
si_E = 130e9
si_nu = 0.28
si_alpha = 2.6e-6

sio2_E = 70e9
sio2_nu = 0.17
sio2_alpha = 0.5e-6

cu_E = 110e9
cu_nu = 0.34
cu_alpha = 16.5e-6
cu_residual_stress_MPa = 900.0
cu_biaxial_modulus = cu_E / (1.0 - cu_nu)
cu_residual_eigenstrain = cu_residual_stress_MPa * 1e6 / cu_biaxial_modulus

unit_layer_specs = [
    {
        "material": "Si",
        "h": si_thickness,
        "E": si_E,
        "nu": si_nu,
        "alpha": si_alpha,
        "epsilon_residual": 0.0,
        "residual_stress_MPa": 0.0,
    },
    {
        "material": "SiO2",
        "h": sio2_thickness,
        "E": sio2_E,
        "nu": sio2_nu,
        "alpha": sio2_alpha,
        "epsilon_residual": 0.0,
        "residual_stress_MPa": 0.0,
    },
    {
        "material": "Cu",
        "h": cu_thickness,
        "E": cu_E,
        "nu": cu_nu,
        "alpha": cu_alpha,
        "epsilon_residual": cu_residual_eigenstrain,
        "residual_stress_MPa": cu_residual_stress_MPa,
    },
]
n_sublayers_per_die = len(unit_layer_specs)

unit_layer_df = pd.DataFrame({
    "material": [spec["material"] for spec in unit_layer_specs],
    "h_um": [spec["h"] * 1e6 for spec in unit_layer_specs],
    "E_GPa": [spec["E"] / 1e9 for spec in unit_layer_specs],
    "nu": [spec["nu"] for spec in unit_layer_specs],
    "alpha_ppm_per_K": [spec["alpha"] * 1e6 for spec in unit_layer_specs],
    "epsilon_residual_ppm": [spec["epsilon_residual"] * 1e6 for spec in unit_layer_specs],
    "residual_stress_MPa": [spec["residual_stress_MPa"] for spec in unit_layer_specs],
})
unit_layer_df["die_unit_total_thickness_um"] = die_total_thickness * 1e6

# Initial die signed warpage distribution parameters.
# Mean and standard deviation are deliberately nonzero/nonidentical to make the yield example meaningful.
mu_W0_all = rng.normal(loc=0.0, scale=3.0e-6, size=N_die)          # [m]
sigma_W0_all = rng.uniform(2.0e-6, 5.0e-6, size=N_die)           # [m]

# Assembly order. Here we use the natural order, but you can replace it with any permutation.
assembly_order = np.arange(N_die)

param_df = pd.DataFrame({
    "die": np.arange(N_die),
    "die_total_thickness_um": die_total_thickness * 1e6,
    "mu_initial_W_um": mu_W0_all * 1e6,
    "sigma_initial_W_um": sigma_W0_all * 1e6,
})

display(unit_layer_df)
param_df


,material,h_um,E_GPa,nu,alpha_ppm_per_K,epsilon_residual_ppm,residual_stress_MPa,die_unit_total_thickness_um
0,Si,48.2,130.0,0.28,2.6,0.0,0.0,50.0
1,SiO2,1.5,70.0,0.17,0.5,0.0,0.0,50.0
2,Cu,0.3,110.0,0.34,16.5,5400.0,900.0,50.0


,die,die_total_thickness_um,mu_initial_W_um,sigma_initial_W_um
0,0,50.0,0.003690,4.986501
1,1,50.0,0.896237,4.377986
2,2,50.0,-0.822414,3.866538
3,3,50.0,-2.671776,4.966880
4,4,50.0,-1.364012,2.645926
5,5,50.0,-2.974940,2.480636
6,6,50.0,0.180431,3.837619
7,7,50.0,4.020646,2.131826
8,8,50.0,-1.476620,2.107041
9,9,50.0,-1.861425,3.544666



## 3. Deterministic warpage and sensitivity matrix

For bonding step $k$, the stack contains the first $k+1$ dies in the assembly order.

The signed warpage at step $k$ can be written as

$$
\widetilde W_k
=
W_{k,\mathrm{det}}
+
\sum_{j=0}^{k} c_{kj} W_{\pi_j}^0,
\qquad k=1,\ldots,N-1.
$$

The sensitivity $c_{kj}$ is computed by unit perturbation:
set $W_{\pi_j}^0=1\,\mu\mathrm{m}$, all other initial warpages to zero, and recompute $\widetilde W_k$.


In [4]:

def build_physical_stack(order_prefix):
    """Build physical Si/SiO2/Cu sublayers for a prefix of assembly_order."""
    h = []
    E = []
    nu = []
    alpha = []
    epsilon_residual = []
    die_position = []
    die_id = []
    material = []

    for local_pos, global_die_id in enumerate(order_prefix):
        for spec in unit_layer_specs:
            h.append(spec["h"])
            E.append(spec["E"])
            nu.append(spec["nu"])
            alpha.append(spec["alpha"])
            epsilon_residual.append(spec["epsilon_residual"])
            die_position.append(local_pos)
            die_id.append(global_die_id)
            material.append(spec["material"])

    return {
        "h": np.asarray(h, dtype=float),
        "E": np.asarray(E, dtype=float),
        "nu": np.asarray(nu, dtype=float),
        "alpha": np.asarray(alpha, dtype=float),
        "epsilon_residual": np.asarray(epsilon_residual, dtype=float),
        "die_position": np.asarray(die_position, dtype=int),
        "die_id": np.asarray(die_id, dtype=int),
        "material": np.asarray(material, dtype=object),
    }


def die_warpage_to_physical_layers(die_W_ordered_prefix):
    """Repeat each die-level initial warpage across its physical sublayers."""
    return np.repeat(np.asarray(die_W_ordered_prefix, dtype=float), n_sublayers_per_die)


def simulate_sequence(assembly_order, die_initial_W_ordered=None, return_aux=False):
    """
    Simulate the sequential stack build.

    The released deterministic bow from each step is carried into the next step.
    Individual die initial warpage is injected only when that die first enters the stack,
    so it is not double-counted in later steps.
    """
    N = len(assembly_order)
    if die_initial_W_ordered is None:
        die_initial_W_ordered = np.zeros(N)
    die_initial_W_ordered = np.asarray(die_initial_W_ordered, dtype=float)

    W_steps = np.zeros(N - 1)
    K_eff_steps = np.zeros(N - 1)
    n_physical_layers_steps = np.zeros(N - 1, dtype=int)
    previous_stack_W = 0.0

    for step in range(1, N):
        # step=1 -> 2 dies assembled.
        prefix = assembly_order[: step + 1]
        layers_step = build_physical_stack(prefix)
        n_physical_layers = len(layers_step["h"])

        # The already released stack curvature is carried into the next process step.
        initial_W = np.full(n_physical_layers, previous_stack_W)

        # The two dies in the first bond enter together. Later, only the newly added die
        # contributes its own pre-bond initial warpage; older die effects are already in
        # previous_stack_W.
        entering_die_W = np.zeros(step + 1)
        if step == 1:
            entering_die_W[0] = die_initial_W_ordered[0]
        entering_die_W[step] = die_initial_W_ordered[step]
        initial_W += die_warpage_to_physical_layers(entering_die_W)

        res = multilayer_warpage(
            layers_step,
            delta_T=delta_T,
            L=L,
            initial_W=initial_W,
        )
        W_steps[step - 1] = res["W"]
        K_eff_steps[step - 1] = res["aux"]["K_eff"]
        n_physical_layers_steps[step - 1] = n_physical_layers
        previous_stack_W = res["W"]

    if return_aux:
        return W_steps, {
            "K_eff_N_m": K_eff_steps,
            "n_physical_layers": n_physical_layers_steps,
        }
    return W_steps


def compute_process_response(assembly_order, delta_T, L, perturb_W=1e-6):
    """
    Compute deterministic step warpages and sensitivity matrix C.

    Returns
    -------
    W_det : ndarray, shape (N_die-1,)
        Deterministic signed warpage after each bonding step [m].
    C : ndarray, shape (N_die-1, N_die)
        Sensitivity matrix. Row i-1 corresponds to bonding step i,
        where i means stack has i+1 dies. Columns follow assembly_order.
    timing : dict
        Timing information.
    aux : dict
        Deterministic stack metadata such as effective bending stiffness.
    """
    t0 = time.perf_counter()

    N = len(assembly_order)
    W_det, aux = simulate_sequence(assembly_order, return_aux=True)
    C = np.zeros((N - 1, N))

    for ordered_j in range(N):
        init_W = np.zeros(N)
        init_W[ordered_j] = perturb_W
        W_j = simulate_sequence(assembly_order, die_initial_W_ordered=init_W)
        C[:, ordered_j] = (W_j - W_det) / perturb_W

    t1 = time.perf_counter()
    return W_det, C, {"response_time_s": t1 - t0}, aux


W_det, C, timing_response, response_aux = compute_process_response(assembly_order, delta_T, L)
abs_W_det_um = np.abs(W_det) * 1e6

K_eff_magnitude_N_m = np.abs(response_aux["K_eff_N_m"])

response_df = pd.DataFrame({
    "bonding_step": np.arange(1, N_die),
    "num_dies_in_stack": np.arange(2, N_die + 1),
    "num_physical_layers": response_aux["n_physical_layers"],
    "W_det_um": W_det * 1e6,
    "abs_W_det_um": abs_W_det_um,
    "delta_abs_W_det_um": np.r_[np.nan, np.diff(abs_W_det_um)],
    "K_eff_magnitude_N_m": K_eff_magnitude_N_m,
})
response_df


,bonding_step,num_dies_in_stack,num_physical_layers,W_det_um,abs_W_det_um,delta_abs_W_det_um,K_eff_magnitude_N_m
0,1,2,6,11.861360,11.861360,NaN,0.007947
1,2,3,9,15.425882,15.425882,3.564522,0.039669
2,3,4,12,17.226802,17.226802,1.800920,0.104688
3,4,5,15,18.327524,18.327524,1.100722,0.214103
4,5,6,18,19.073674,19.073674,0.746150,0.379014
5,6,7,21,19.614091,19.614091,0.540417,0.610519
6,7,8,24,20.024073,20.024073,0.409982,0.919718
7,8,9,27,20.345997,20.345997,0.321923,1.317710
8,9,10,30,20.605601,20.605601,0.259604,1.815594
9,10,11,33,20.819449,20.819449,0.213848,2.424469


In [5]:

print("Timing for deterministic warpage + sensitivity matrix:")
print(timing_response)

plt.figure(figsize=(7, 4))
plt.plot(response_df["num_dies_in_stack"], response_df["abs_W_det_um"], marker="o")
plt.xlabel("Number of dies in stack")
plt.ylabel("Deterministic bow magnitude (um)")
plt.title("Bow magnitude after each bonding step")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(
    response_df["num_dies_in_stack"].iloc[1:],
    response_df["delta_abs_W_det_um"].iloc[1:],
    marker="o",
)
plt.xlabel("Number of dies in stack")
plt.ylabel("Incremental bow magnitude (um)")
plt.title("Incremental bow decreases as stack stiffness increases")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(response_df["num_dies_in_stack"], response_df["K_eff_magnitude_N_m"], marker="o")
plt.xlabel("Number of dies in stack")
plt.ylabel("Effective bending stiffness magnitude per unit width (N m)")
plt.title("Stack stiffness magnitude increases with each added die unit")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.imshow(C, aspect="auto")
plt.colorbar(label="Sensitivity c_ij")
plt.xlabel("Die position in assembly order")
plt.ylabel("Bonding step index")
plt.title("Sensitivity matrix C")
plt.show()


Timing for deterministic warpage + sensitivity matrix:
{'response_time_s': 0.05486553907394409}



## 4. Paper Eq. (1)-(3) check without initial warpage

The paper-style calculation below follows the equations in the screenshot. I interpret the term printed as $\Delta\sigma_i\Delta t$ as the thermal mismatch term $\Delta\alpha_i\Delta T$.
With the residual-eigenstrain option enabled, the free-strain difference is written more generally as

$$
\Delta\epsilon_i
=
\left(\alpha_i-\alpha_0\right)\Delta T
+
\left(\epsilon_i^{\mathrm{res}}-\epsilon_0^{\mathrm{res}}\right).
$$

For this check, layer 0 is the bottom Si sublayer of the current physical stack, and all other physical sublayers are treated as films $i=1,\ldots,m$.
No random initial die warpage and no previous-step bow carry-over are included here; each stack height is solved as a fresh stack using only Eq. (1)-(3).


In [6]:

def paper_formula_warpage(layers, delta_T, x):
    """
    Implement the paper-style Eq. (1)-(3) warpage calculation.

    Layer 0 is treated as the substrate. The printed Delta sigma_i term is
    interpreted as Delta alpha_i * Delta T, generalized here as the free-strain
    difference relative to layer 0.
    """
    h = np.asarray(layers["h"], dtype=float)
    E = np.asarray(layers["E"], dtype=float)
    nu = np.asarray(layers["nu"], dtype=float)
    alpha = np.asarray(layers["alpha"], dtype=float)
    epsilon_residual = np.asarray(
        layers.get("epsilon_residual", np.zeros_like(h)), dtype=float
    )

    if len(h) < 2:
        return {
            "W_m": 0.0,
            "curvature_1_per_m": 0.0,
            "F0_N_per_m": 0.0,
            "film_force_sum_N_per_m": 0.0,
            "eq1_minus_curvature_W_m": 0.0,
        }

    E0 = E / (1.0 - nu)
    axial_stiffness = E0 * h
    lambda_0 = 1.0 / axial_stiffness[0]

    free_strain = alpha * delta_T + epsilon_residual
    delta_free_strain = free_strain - free_strain[0]

    a = np.cumsum(h) - 0.5 * (h[0] + h)

    # Films only, i = 1, ..., m.
    Af = axial_stiffness[1:]
    df = delta_free_strain[1:]
    af = a[1:]

    S_lambda0 = np.sum(Af * lambda_0)
    S_free = np.sum(Af * df)
    S_a = np.sum(Af * af)

    # Eq. (1) is equivalent to kappa = 6 / (E_s^0 h_s^2) * sum_i F_i.
    curvature_factor = 6.0 / (E0[0] * h[0] ** 2)

    # Unknowns are F0 and kappa = 1 / rho.
    # Force balance: F0 + sum_i F_i = 0.
    # Curvature relation: kappa - curvature_factor * sum_i F_i = 0.
    system_matrix = np.array([
        [1.0 + S_lambda0, -S_a],
        [-curvature_factor * S_lambda0, 1.0 + curvature_factor * S_a],
    ])
    rhs = np.array([-S_free, curvature_factor * S_free])
    F0, curvature = np.linalg.solve(system_matrix, rhs)

    film_force_sum = S_lambda0 * F0 + S_free - S_a * curvature
    W_from_curvature = curvature * x ** 2 / 2.0
    W_from_eq1 = 3.0 / E0[0] * (x / h[0]) ** 2 * film_force_sum

    return {
        "W_m": float(W_from_curvature),
        "curvature_1_per_m": float(curvature),
        "F0_N_per_m": float(F0),
        "film_force_sum_N_per_m": float(film_force_sum),
        "eq1_minus_curvature_W_m": float(W_from_eq1 - W_from_curvature),
    }


def paper_formula_stack_sweep(include_cu_residual):
    rows = []
    for n_die in range(1, N_die + 1):
        layers = build_physical_stack(np.arange(n_die))
        if not include_cu_residual:
            layers = dict(layers)
            layers["epsilon_residual"] = np.zeros_like(layers["epsilon_residual"])

        out = paper_formula_warpage(layers, delta_T=delta_T, x=L)
        rows.append({
            "case": "thermal + Cu residual" if include_cu_residual else "thermal only",
            "num_dies_in_stack": n_die,
            "num_physical_layers": len(layers["h"]),
            "W_paper_um": out["W_m"] * 1e6,
            "abs_W_paper_um": abs(out["W_m"] * 1e6),
            "curvature_1_per_m": out["curvature_1_per_m"],
            "eq1_minus_curvature_um": out["eq1_minus_curvature_W_m"] * 1e6,
        })
    return pd.DataFrame(rows)

paper_formula_df = pd.concat(
    [
        paper_formula_stack_sweep(include_cu_residual=False),
        paper_formula_stack_sweep(include_cu_residual=True),
    ],
    ignore_index=True,
)
paper_formula_df["delta_abs_W_paper_um"] = paper_formula_df.groupby("case")["abs_W_paper_um"].diff()

paper_formula_df


,case,num_dies_in_stack,num_physical_layers,W_paper_um,abs_W_paper_um,curvature_1_per_m,eq1_minus_curvature_um,delta_abs_W_paper_um
0,thermal only,1,3,-19.480193,19.480193,-1.558415,6.776264e-15,NaN
1,thermal only,2,6,-4.956773,4.956773,-0.396542,3.388132e-15,-14.523420
2,thermal only,3,9,-2.839666,2.839666,-0.227173,8.046813e-15,-2.117107
3,thermal only,4,12,-1.989796,1.989796,-0.159184,5.082198e-15,-0.849870
4,thermal only,5,15,-1.531455,1.531455,-0.122516,-5.505714e-15,-0.458342
5,thermal only,6,18,-1.244735,1.244735,-0.099579,-1.927000e-14,-0.286720
6,thermal only,7,21,-1.048445,1.048445,-0.083876,-2.689330e-14,-0.196290
7,thermal only,8,24,-0.905630,0.905630,-0.072450,-3.483423e-14,-0.142815
8,thermal only,9,27,-0.797058,0.797058,-0.063765,1.164670e-14,-0.108572
9,thermal only,10,30,-0.711732,0.711732,-0.056939,2.583450e-14,-0.085326


In [7]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for case, sub in paper_formula_df.groupby("case"):
    axes[0].plot(sub["num_dies_in_stack"], sub["abs_W_paper_um"], marker="o", label=case)
    axes[1].plot(
        sub["num_dies_in_stack"].iloc[1:],
        sub["delta_abs_W_paper_um"].iloc[1:],
        marker="o",
        label=case,
    )

axes[0].set_xlabel("Number of dies in stack")
axes[0].set_ylabel("Paper-formula bow magnitude (um)")
axes[0].set_title("Fresh-stack Eq. (1)-(3) result")
axes[0].grid(True)
axes[0].legend()

axes[1].set_xlabel("Number of dies in stack")
axes[1].set_ylabel("Incremental bow magnitude (um)")
axes[1].set_title("Increment relative to previous stack count")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()



## 5. Analytical warpage yield from multivariate Gaussian

Given

$$
\widetilde{\mathbf W}
=
\mathbf W_{\mathrm{det}}
+
\mathbf C\mathbf W^0_{\pi}
$$

and independent Gaussian initial warpages,

$$
\mathbf W^0_{\pi}\sim
\mathcal{N}\!\left(\boldsymbol{\mu}_{0,\pi},\boldsymbol{\Sigma}_{0,\pi}\right)
$$

we have

$$
\widetilde{\mathbf W}
\sim
\mathcal{N}
\!\left(\boldsymbol{\mu}_{W},\boldsymbol{\Sigma}_{W}\right)
$$

where

$$
\begin{aligned}
\boldsymbol{\mu}_{W}
&=
\mathbf W_{\mathrm{det}}
+
\mathbf C\boldsymbol{\mu}_{0,\pi},\\
\boldsymbol{\Sigma}_{W}
&=
\mathbf C\boldsymbol{\Sigma}_{0,\pi}\mathbf C^{T}.
\end{aligned}
$$

The stack survives if

$$
\left|\widetilde W_k\right|<W_{\mathrm{th}},\qquad k=1,\ldots,N-1.
$$

So

$$
Y_{\mathrm{stack}}^{(\mathrm{warp})}
=
\Pr\!\left(
-W_{\mathrm{th}}\mathbf 1
<
\widetilde{\mathbf W}
<
W_{\mathrm{th}}\mathbf 1
\right).
$$


In [8]:

def analytical_gaussian_yield(W_det, C, mu_W0, sigma_W0, W_th, cdf_seed=123):
    """
    Compute the stack warpage yield using a multivariate normal rectangle probability.

    Returns
    -------
    yield_value : float
    mu_W : ndarray
    cov_W : ndarray
    timing : dict
    """
    t0 = time.perf_counter()

    Sigma0 = np.diag(sigma_W0**2)
    mu_W = W_det + C @ mu_W0
    cov_W = C @ Sigma0 @ C.T

    lower = -W_th * np.ones_like(W_det)
    upper = W_th * np.ones_like(W_det)

    # SciPy's multivariate_normal.cdf supports lower_limit in the repo environment.
    # Keep cdf_seed in the function signature for notebook compatibility; SciPy 1.13
    # does not expose an rng keyword here.
    yield_value = multivariate_normal.cdf(
        upper,
        mean=mu_W,
        cov=cov_W,
        allow_singular=True,
        lower_limit=lower,
        maxpts=200000,
        abseps=1e-6,
        releps=1e-6,
    )

    t1 = time.perf_counter()
    return yield_value, mu_W, cov_W, {"analytic_cdf_time_s": t1 - t0}

# Reorder distribution parameters according to assembly order.
mu_W0_order = mu_W0_all[assembly_order]
sigma_W0_order = sigma_W0_all[assembly_order]

yield_analytic, mu_W, cov_W, timing_analytic = analytical_gaussian_yield(
    W_det, C, mu_W0_order, sigma_W0_order, W_th
)

print(f"Analytical multivariate Gaussian yield = {yield_analytic:.6f}")
print("Timing:", timing_analytic)

dist_df = pd.DataFrame({
    "bonding_step": np.arange(1, N_die),
    "num_dies_in_stack": np.arange(2, N_die + 1),
    "mean_step_W_um": mu_W * 1e6,
    "std_step_W_um": np.sqrt(np.diag(cov_W)) * 1e6,
    "threshold_um": W_th * 1e6,
})
dist_df


Analytical multivariate Gaussian yield = 0.373133
Timing: {'analytic_cdf_time_s': 0.02892959862947464}


,bonding_step,num_dies_in_stack,mean_step_W_um,std_step_W_um,threshold_um
0,1,2,12.301327,3.328262,20.0
1,2,3,15.672104,3.450658,20.0
2,3,4,17.095160,3.521432,20.0
3,4,5,18.066768,3.530327,20.0
4,5,6,18.610984,3.534341,20.0
5,6,7,19.160618,3.539773,20.0
6,7,8,19.730811,3.540793,20.0
7,8,9,20.005548,3.541433,20.0
8,9,10,20.216382,3.542650,20.0
9,10,11,20.462373,3.543430,20.0



## 6. Monte Carlo validation

Monte Carlo samples the initial die warpages, propagates them through the sensitivity matrix, and estimates yield as the fraction of samples satisfying all step-wise constraints:

$$
\widehat Y_{\mathrm{MC}}
=
\frac{1}{N_{\mathrm{MC}}}
\sum_{\ell=1}^{N_{\mathrm{MC}}}
\mathbf 1\!\left[
\left|\widetilde W_k^{(\ell)}\right|<W_{\mathrm{th}},\ \forall k
\right].
$$


In [9]:

def monte_carlo_yield(W_det, C, mu_W0, sigma_W0, W_th, n_samples=1_000_000, seed=2026):
    """
    Vectorized Monte Carlo validation for stack warpage yield.

    Returns
    -------
    yield_mc : float
    se_mc : float
        Standard error estimate of MC yield.
    timing : dict
    """
    t0 = time.perf_counter()
    rng_mc = np.random.default_rng(seed)

    # Sample initial signed warpages of all dies.
    W0_samples = rng_mc.normal(
        loc=mu_W0,
        scale=sigma_W0,
        size=(n_samples, len(mu_W0)),
    )

    # Propagate to step warpages.
    # W_steps shape: (n_samples, N_die-1)
    W_steps = W_det[None, :] + W0_samples @ C.T

    survive = np.all(np.abs(W_steps) < W_th, axis=1)
    yield_mc = np.mean(survive)
    se_mc = np.sqrt(yield_mc * (1.0 - yield_mc) / n_samples)

    t1 = time.perf_counter()
    return yield_mc, se_mc, {"mc_time_s": t1 - t0, "n_samples": n_samples}

# Use 1e6 samples by default. Reduce this if your machine is slow.
yield_mc, se_mc, timing_mc = monte_carlo_yield(
    W_det, C, mu_W0_order, sigma_W0_order, W_th, n_samples=1_000_000
)

print(f"Monte Carlo yield = {yield_mc:.6f} ± {1.96*se_mc:.6f} (95% CI)")
print("Timing:", timing_mc)
print(f"Analytical yield = {yield_analytic:.6f}")
print(f"Absolute difference = {abs(yield_mc - yield_analytic):.6f}")


Monte Carlo yield = 0.373656 ± 0.000948 (95% CI)
Timing: {'mc_time_s': 0.8003575652837753, 'n_samples': 1000000}
Analytical yield = 0.373133
Absolute difference = 0.000523



## 7. Timing summary


In [10]:

timing_summary = pd.DataFrame([
    {"stage": "deterministic warpage + sensitivity matrix", **timing_response},
    {"stage": "analytical multivariate Gaussian CDF", **timing_analytic},
    {"stage": "Monte Carlo validation", **timing_mc},
])
timing_summary


,stage,response_time_s,analytic_cdf_time_s,mc_time_s,n_samples
0,deterministic warpage + sensitivity matrix,0.054866,NaN,NaN,NaN
1,analytical multivariate Gaussian CDF,NaN,0.02893,NaN,NaN
2,Monte Carlo validation,NaN,NaN,0.800358,1000000.0


In [11]:

plt.figure(figsize=(7, 4))
plt.errorbar(
    ["Analytical MVN CDF", "Monte Carlo"],
    [yield_analytic, yield_mc],
    yerr=[0, 1.96 * se_mc],
    fmt="o",
    capsize=5,
)
plt.ylim(0, 1.05)
plt.ylabel("Warpage yield")
plt.title("Analytical yield vs Monte Carlo validation")
plt.grid(True)
plt.show()



## 8. Notes

- The deterministic warpage $W_{k,\mathrm{det}}$ is computed by setting all initial die warpages to zero.
- The sensitivity matrix $C$ is computed by unit perturbation, so the method works even if the layer thicknesses and physical parameters are all different.
- The analytical yield uses the joint multivariate Gaussian distribution of all step warpages.
- The Monte Carlo validation should agree with the analytical yield within the Monte Carlo confidence interval.
- For assembly-order optimization, wrap this notebook's response/yield calculation inside a loop over candidate orders.
